In [ ]:
from src.utils import setup_client
import requests
import numpy as np
import pandas as pd
import re
import math

### Run the cells below to set up client and load query function

In [ ]:
hf_client = setup_client()

In [ ]:
def query_model(prompt, model_name, max_tokens, client):
    """
    Parameters
    ----------
    prompt     : str   — the input text
    model_name : str   — "gpt2", "llama-1b", or "llama-8b"
    max_tokens : int   — how many tokens to generate (capped at 200 by proxy)
    client     : str   — the proxy URL returned by setup_client()

    Returns
    -------
    dict with keys:
        "answer"      : str   — the generated text
        "logprobs"    : dict  — {token: logprob}
        "token_probs" : dict  — {token: probability}
    """
    response = requests.post(
        f"{client}/generate",
        json={
            "prompt": prompt,
            "model": model_name,
            "max_tokens": max_tokens,
        },
        timeout=60,
    )

    if response.status_code == 429:
        print("⏳ Rate limit hit — wait a moment and try again.")
        return None

    if response.status_code == 400:
        print(f"❌ Bad request: {response.json().get('detail')}")
        return None

    response.raise_for_status()
    return response.json()

## Exercise 1 - Generate **Open Question** test sets
For the exercise below, we will need open questions instead of MCQs. We ask you to generate two question sets with each 5 questions. The questions must range from easy to difficult questions for an LLM to answer. The first question set you must generate yourself. Think carefully: which questions are easy to answer for an AI, and which are the most difficult? Range them from easy to difficult. When you're done, ask an LLM of your choice to generate a second test set with 5 questions, also ranging from easy to difficult to answer for that LLM. Save the question sets in two lists in the cell below.

In [ ]:
self_made_test_set = [] #TO DO
generated_test_set = [] #TO DO

## Exercise 2 - Color coding answer of LLM
In this exercise, we will color code the answers of an LLM to your open questions based on the token probabilities of the answer. We will use some predefined helper functions listed below. First study these functions briefly and try to understand what they do. 

In [ ]:
# DEFINING HELPER FUNCTIONS
def hex_to_rgb(hex_code):
    """Converts a 6-digit hex string (for example: 'FF0000') to an RGB tuple"""
    r = int(hex_code[0:2], 16)
    g = int(hex_code[2:4], 16)
    b = int(hex_code[4:6], 16)
    return r, g, b

def colored(text, bg_hex):
    """
    Applies the given background hex color to text (using 'ANSI 24-bit')
    A black foreground (000000) is used for contrast
    """
    fg_r, fg_g, fg_b = 0, 0, 0  # Black foreground
    bg_r, bg_g, bg_b = hex_to_rgb(bg_hex)

    # ANSI escape code: (1) means bold and (48;2) background and (38;2) foreground
    # create formatted string and apply the background color to the text
    return f"\033[1;38;2;{fg_r};{fg_g};{fg_b};48;2;{bg_r};{bg_g};{bg_b}m{text}\033[0m"

def prob_to_color(avg_prob):
    """
    Maps the avg standard probability (a value between 0.0 and 1.0) of the sentence to a smooth hex color 
    gradient from red (0.0) to green (1.0)
    """
    # Clamp the input probability between 0.0 and 1.0 to avoid pointing float errors
    norm_prob = np.clip(avg_prob, 0.0, 1.0)
    
    # Red value decreases as probability increases
    r = int((1 - norm_prob) * 255)
    # Green value increases as probability increases 
    g = int(norm_prob * 255)
    # Blue value remains 0
    b = 0
    
    return f"{r:02x}{g:02x}{b:02x}"

**Try it out!**

In [ ]:
sentence = "The color of this sentence shows how confident I am that I can finish this exercise correctly"
confidence = # TO DO rate your confidence
bg_color_hex = # TO DO
colored_sentence = # TO DO (tip: use f" " for formatting your string)
print(colored_sentence)


### Investigating response structure

Now we will write a function that can process an answer of an LLM and color code it based on the average token probability *per sentence*. First we will start by investigating the response structure of the LLM. Run the cell below and examine the datastructures. 

In [ ]:
prompt = 'If AI could dream, what would they dream about? Answer in a max of 5 sentences.'
model_name = "llama-8b"
max_tokens = 200
client = hf_client

response = query_model(prompt, model_name, max_tokens, client)
answer = response['answer']
logprobs = response['logprobs'] 
token_probs = response['token_probs']
logprobs_content = response['logprobs_content']

print(f"ANSWER: \n{answer}\n")
print('-'*100)
print(f"LOGPROBS: \n{logprobs}\n")
print('-'*100)
print(f"TOKEN_PROBS: \n{token_probs}\n")
print('-'*100)
print(f"LOGPROBS_CONTENT: \n{logprobs_content}\n")

### 2A) Finish 'color_code_per_sentence()'
Now scan the function below. Which datastructure would be best for this function? Choose the best option and finish the function by filling out the gaps.  

In [ ]:
token_data =                                      # TO DO select the best datastructure

In [ ]:
def color_code_per_sentence(token_data):
    """
    Color codes the sentences of an LLM answer based on average token probability.
    
    Parameters
    ----------
    token_data : the input datastructure that contains the tokens and their token probabilities
    """
    colored_output = [] # to store the final output

    # we will build back each sentence by iterating over the input data
    # we also calculate the avg_prob per sentence
    # based on these avg_probs, we will give the sentence a background color
    current_sentence = ""
    current_sentence_probs = [] # to store the avg_prob values
    
    for item in token_data:
        token = item['token']
        logprob = item['logprob']
        
        # Skip hidden stop tokens like <|eot_id|>
        if token == "<|eot_id|>":
            continue
            
        # Convert logprob to normal probability
        prob = math.exp(logprob)
        
        # Keep adding tokens to the current sentence
        # Also keep track of their probabilities
        current_sentence += token
        current_sentence_probs.append(prob)
        
        # We stop adding tokens to the sentence if the sentence ends
        # So check if the token contains a sentence-ending punctuation mark
        if any(punct in token for punct in ['.', '?', '!']):
            
            # Calculate average probability for this specific sentence
            avg_prob =                                                  # TO DO
            
            # Map the average probability to a hex color
            bg_color_hex =                                              # TO DO
            prob_percent = f"{avg_prob * 100:.2f}%"
            
            # Clean up leading spaces for printing and apply color
            clean_sentence =                                            # TO DO
            colored_sentence = f"{...} {prob_percent}"                  # TO DO 
            
            # Save to output
            colored_output.append(...)                                  # TO DO
            
            # Reset lists for the next sentence
            current_sentence =                                          # TO DO 
            current_sentence_probs =                                    # TO DO

    # Catch-all for any leftover text at the end that didn't end in punctuation
    if current_sentence.strip():
        avg_prob =                                                      # TO DO
        bg_color_hex =                                                  # TO DO                                                
        prob_percent = f"{avg_prob * 100:.2f}%"
        
        clean_sentence =                                                # TO DO
        colored_sentence = f"{} {prob_percent}"                         # TO DO

        # Save to output
                                                                        # TO DO

    print("--- Colored Confidence Output ---")
    print("\n".join(colored_output))
    print("-" * 50)

#### Test color_code_per_sentence()
Test the function by filling out the chosen datastructure below. Then run it. 

In [ ]:
token_data =                                                # TO DO
color_code_per_sentence(token_data)

### 2B) Finish 'color_code_per_token()'
Now color code the answer of the LLM per token instead of per sentence. Finish the function below

In [ ]:
def color_code_per_token(token_data):
    """
    Color codes an LLM answer based on individual token probabilities.
    
    Parameters
    ----------
    token_data : containing the tokens and their logprobs
    """
    colored_tokens = [] # to store the individually colored tokens

    for item in token_data:
        token = item['token']
        logprob = item['logprob']
        
        # Skip hidden stop tokens like <|eot_id|>
        if token == "<|eot_id|>":
            continue
            
        # Convert logprob to normal probability
        prob =                                                            # TO DO
        
        # Map the probability to a hex color
        bg_color_hex =                                                    # TO DO
        
        # Apply the color directly to the token
        colored_token =                                                   # TO DO
        
        # Save to output
        colored_tokens.append(...)                                        # TO DO
    print("--- Colored Confidence Output (Per Token) ---")
    
    print(...)                                                            # TO DO
    
    print("-" * 50)

#### Test color_code_per_token()
Run the cell below

In [ ]:
color_code_per_token(token_data)

### Exercise 2C) Test your function on the open question test sets! 
Run the cells below. **Stop and think:** what can you say about the results? Was the LLM able to estimate which questions were difficult? Were you able to? What do the token probabilities mean in this context? Write down anything else that you notice about the results. 

Note: **only run the cells below if you tested your functions** in part 2A and 2B and it worked.

**Write your answer in the cell below:**

In [ ]:
# My answer:

In [ ]:
model_name = "llama-8b"
max_tokens = 200
client = hf_client

for question in generated_test_set:
    prompt = question + ' Answer in a maximum of 5 sentences.'
    response = query_model(prompt, model_name, max_tokens, client)
    token_data = response['logprobs_content']
    color_code_per_sentence(token_data)

In [ ]:
for question in generated_test_set:
    prompt = question + ' Answer in a maximum of 5 sentences.'
    response = query_model(prompt, model_name, max_tokens, client)
    token_data = response['logprobs_content']
    color_code_per_token(token_data)

In [ ]:
for question in self_made_test_set:
    prompt = question + ' Answer in a maximum of 5 sentences.'
    response = query_model(prompt, model_name, max_tokens, client)
    token_data = response['logprobs_content']
    color_code_per_sentence(token_data)

In [ ]:
for question in self_made_test_set:
    prompt = question + ' Answer in a maximum of 5 sentences.'
    response = query_model(prompt, model_name, max_tokens, client)
    token_data = response['logprobs_content']
    color_code_per_token(token_data)